# EfficientNetB4 + XGBoost — Garlic Disease Classification (Multi-Run Experiment)

**Strategy:** Fine-tune Top-5 Blocks → GlobalAveragePooling (1792-dim) → XGBoost Classifier

| Item | Value |
|---|---|
| **Architecture** | EfficientNetB4 (fine-tuned blocks 3-4-5-6-7) + XGBoost |
| **Input size** | 380 × 380 × 3 |
| **Pipeline** | Image → EfficientNetB4 backbone → GAP (1792-dim) → XGBoost |
| **Strategy** | 3-seed independent runs for statistical robustness |
| **Optimizer (CNN)** | Adam + ExponentialDecay (lr=1e-4) |
| **Loss (CNN)** | CategoricalCrossentropy (label_smoothing=0.15) |
| **Classifier** | XGBoost (n_estimators=500, max_depth=6, lr=0.05) |
| **Regularisation** | Dropout 0.5 + L2 1e-5 + Class-weight balancing |

---

In [ ]:
# ========== 1. IMPORTS ========== #
import os
import glob
import time
import random
import shutil
import tempfile
import pickle
from collections import defaultdict
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm_lib
import seaborn as sns
import tensorflow as tf
import xgboost as xgb

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.regularizers import l2

from sklearn.utils import class_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                              top_k_accuracy_score, roc_curve, auc,
                              cohen_kappa_score, matthews_corrcoef,
                              balanced_accuracy_score)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

# ========== 2. GPU CONFIG ========== #
print("TensorFlow version:", tf.__version__)
print("XGBoost version   :", xgb.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", len(gpus))
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled.")
    except RuntimeError as e:
        print(e)

# ========== 3. MIXED PRECISION ========== #
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Compute dtype :", tf.keras.mixed_precision.global_policy().compute_dtype)
print("Variable dtype:", tf.keras.mixed_precision.global_policy().variable_dtype)

In [ ]:
# ========== 4. EXPERIMENT CONFIGURATION ========== #

# ── STRATEGY ──────────────────────────────────────────────────────────────
STRATEGY_KEY    = "finetune_top5_xgb"
STRATEGY_LABEL  = "Fine-tune Top-5 + XGBoost (blocks 3-4-5-6-7)"
UNFREEZE_BLOCKS = [3, 4, 5, 6, 7]
USE_AUG         = True
LR              = 1e-4
# ──────────────────────────────────────────────────────────────────────────

# --- XGBoost hyperparameters ---
XGB_PARAMS = dict(
    n_estimators          = 500,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 3,
    gamma                 = 0.1,
    reg_alpha             = 0.1,
    reg_lambda            = 1.0,
    tree_method           = 'hist',
    device                = 'cuda',   # falls back to CPU if no GPU
    eval_metric           = 'mlogloss',
    early_stopping_rounds = 30,
)

# --- Paths ---
DATA_DIR        = "/kaggle/input/datasets/giaphuc/dataset-garlic-2106/dataset_final_2006"
BASE_RESULT_DIR = f"/kaggle/working/report_EfficientNetB4/{STRATEGY_KEY}"
os.makedirs(BASE_RESULT_DIR, exist_ok=True)

# --- Model ---
INPUT_SHAPE = (380, 380, 3)
# blocks 1-2 frozen → no 190×190 backprop activations → safe at batch=64
BATCH_SIZE         = 64
# Feature extraction runs the full backbone without gradients → more VRAM pressure per image
# Use a smaller batch to avoid ResourceExhaustedError during extract_features()
EXTRACT_BATCH_SIZE = 16
EPOCHS             = 30

# --- Shared hyperparameters ---
LABEL_SMOOTHING = 0.15
DROPOUT_RATE    = 0.5
PATIENCE        = 12

# --- Multi-run settings ---
# Set N_RUNS = 1 for a single training run (fastest), 2 or 3 for statistical robustness.
# Only the first N_RUNS seeds from RANDOM_SEEDS will be used.
N_RUNS       = 3
RANDOM_SEEDS = [42, 123, 456]

# --- Performance knobs ---
AUTOTUNE = tf.data.AUTOTUNE
tf.config.optimizer.set_jit(True)

# --- Result storage ---
all_runs_results = []

print(f"Strategy   : {STRATEGY_LABEL}  (key={STRATEGY_KEY})")
print(f"Unfreeze   : {UNFREEZE_BLOCKS}  |  Augmentation: {USE_AUG}  |  LR: {LR}")
print(f"Dataset    : {DATA_DIR}")
print(f"Output dir : {BASE_RESULT_DIR}")
print(f"Input shape: {INPUT_SHAPE}")
print(f"Batch size : {BATCH_SIZE}  |  Extract batch: {EXTRACT_BATCH_SIZE}")
print(f"N_RUNS     : {N_RUNS}  |  Seeds: {RANDOM_SEEDS[:N_RUNS]}")
print(f"XLA JIT    : ON")
print(f"XGB params : n_estimators={XGB_PARAMS['n_estimators']}  max_depth={XGB_PARAMS['max_depth']}  lr={XGB_PARAMS['learning_rate']}")

In [ ]:
# ========== 5. HELPER FUNCTIONS ========== #

# ---------- GPU-side augmentation ----------
_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.083),
    tf.keras.layers.RandomZoom(0.20),
    tf.keras.layers.RandomTranslation(0.20, 0.20),
    tf.keras.layers.RandomBrightness(factor=0.30),
], name='augmentation')


def apply_freeze_strategy(base, unfreeze_blocks):
    """Selectively unfreeze EfficientNetB4 blocks (BN always frozen)."""
    base.trainable = False
    if unfreeze_blocks == "all":
        for layer in base.layers:
            if not isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = True
    elif isinstance(unfreeze_blocks, list) and len(unfreeze_blocks) > 0:
        for layer in base.layers:
            for block_num in unfreeze_blocks:
                if layer.name.startswith(f"block{block_num}"):
                    if not isinstance(layer, tf.keras.layers.BatchNormalization):
                        layer.trainable = True
                    break
    trainable = sum(1 for l in base.layers if l.trainable)
    total     = len(base.layers)
    print(f"  [{STRATEGY_LABEL}] {trainable}/{total} base layers trainable (BN always frozen)")


def _collect_samples(split_dir, class_to_idx):
    """Walk split_dir → (abs_paths, int_labels, rel_filenames), sorted."""
    paths, labels, filenames = [], [], []
    for cn, ci in sorted(class_to_idx.items()):
        d = os.path.join(split_dir, cn)
        for fname in sorted(os.listdir(d)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                paths.append(os.path.join(d, fname))
                labels.append(ci)
                filenames.append(f"{cn}/{fname}")
    return paths, labels, filenames


def create_tf_datasets(data_dir, input_shape, batch_size, seed=None, use_aug=True):
    """Build GPU-optimised tf.data pipelines (one-hot labels for backbone training)."""
    class_names  = sorted([
        d for d in os.listdir(os.path.join(data_dir, 'train'))
        if os.path.isdir(os.path.join(data_dir, 'train', d))
    ])
    class_to_idx = {cn: i for i, cn in enumerate(class_names)}
    num_classes  = len(class_names)
    h, w         = input_shape[:2]

    def load_and_preprocess(path, label):
        raw   = tf.io.read_file(path)
        img   = tf.image.decode_jpeg(raw, channels=3)
        img   = tf.image.resize(img, [h, w])
        img   = tf.cast(img, tf.float32)
        img   = efficientnet_preprocess(img)
        label = tf.one_hot(label, depth=num_classes)
        return img, label

    def augment(img, lbl):
        return _augmentation(img, training=True), lbl

    def _make_split(split, training=False, apply_augmentation=False):
        sdir               = os.path.join(data_dir, split)
        paths, labels, fns = _collect_samples(sdir, class_to_idx)
        n  = len(paths)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(n, seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
        if apply_augmentation:
            ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=training)
        ds = ds.prefetch(AUTOTUNE)
        return ds, n, fns, labels

    train_ds, n_train, _,           train_lbl  = _make_split('train', training=True,  apply_augmentation=use_aug)
    val_ds,   n_val,   _,           _          = _make_split('val',   training=False, apply_augmentation=False)
    test_ds,  n_test,  test_fnames, test_lbl   = _make_split('test',  training=False, apply_augmentation=False)

    cw      = class_weight.compute_class_weight(
        'balanced', classes=np.unique(train_lbl), y=train_lbl)
    cw_dict = dict(enumerate(cw))

    meta = SimpleNamespace(
        class_names       = class_names,
        class_to_idx      = class_to_idx,
        num_classes       = num_classes,
        test_filenames    = test_fnames,
        test_classes      = np.array(test_lbl),
        n_train           = n_train,
        n_val             = n_val,
        n_test            = n_test,
        class_weight_dict = cw_dict,
    )
    aug_info = "ON" if use_aug else "OFF"
    print(f"  Augmentation: {aug_info}  |  train={n_train}  val={n_val}  test={n_test}")
    return train_ds, val_ds, test_ds, meta


def extract_features(feature_extractor, data_dir, split, class_to_idx, input_shape, batch_size):
    """Extract 1792-dim GAP features (no augmentation, integer labels for XGBoost)."""
    paths, labels, _ = _collect_samples(os.path.join(data_dir, split), class_to_idx)
    h, w = input_shape[:2]
    ds = (tf.data.Dataset.from_tensor_slices((paths, labels))
          .map(lambda p, l: (
              efficientnet_preprocess(tf.cast(tf.image.resize(
                  tf.image.decode_jpeg(tf.io.read_file(p), channels=3), [h, w]
              ), tf.float32)), l
          ), num_parallel_calls=AUTOTUNE)
          .batch(batch_size)
          .prefetch(AUTOTUNE))
    all_feats, all_labels = [], []
    for batch_x, batch_y in ds:
        feats = feature_extractor(batch_x, training=False)
        all_feats.append(feats.numpy())
        all_labels.append(batch_y.numpy())
    return np.concatenate(all_feats).astype(np.float32), np.concatenate(all_labels).astype(np.int32)


def build_model(input_shape, num_classes, steps_per_epoch):
    """EfficientNetB4 + custom head for backbone fine-tuning."""
    base = EfficientNetB4(weights='imagenet', include_top=False, input_shape=input_shape)
    apply_freeze_strategy(base, UNFREEZE_BLOCKS)
    x   = GlobalAveragePooling2D()(base.output)
    x   = BatchNormalization()(x)
    x   = Dense(128, activation='relu', kernel_regularizer=l2(1e-5))(x)
    x   = Dropout(DROPOUT_RATE)(x)
    out = Dense(num_classes, activation='softmax', dtype='float32')(x)
    model = Model(inputs=base.input, outputs=out)
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=LR,
        decay_steps=steps_per_epoch * 5,
        decay_rate=0.9,
        staircase=True,
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss=CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=['accuracy'],
    )
    return model


print("Helper functions defined.")
print(f"  apply_freeze_strategy — top-down block unfreezing (BN always frozen)")
print(f"  create_tf_datasets    — GPU-optimised tf.data pipeline (one-hot labels)")
print(f"  extract_features      — GAP feature extraction (integer labels, no aug)")
print(f"  build_model           — EfficientNetB4 | LR={LR} | dropout={DROPOUT_RATE}")

In [ ]:
# ========== 6. MULTI-RUN TRAINING: EfficientNetB4 → GAP → XGBoost ========== #
import csv

for run_idx, seed in enumerate(RANDOM_SEEDS[:N_RUNS]):
    print("\n" + "="*70)
    print(f" RUN {run_idx+1}/{N_RUNS}  —  seed={seed}")
    print(f" Strategy : {STRATEGY_LABEL}  |  LR={LR}  |  Aug={USE_AUG}")
    print("="*70)

    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

    RESULT_DIR = os.path.join(BASE_RESULT_DIR, f"run_{run_idx+1}_seed_{seed}")
    os.makedirs(RESULT_DIR, exist_ok=True)

    # ── STEP 1: Fine-tune EfficientNetB4 backbone ──────────────────────────
    print("\n[Step 1/3] Fine-tuning EfficientNetB4 backbone (blocks 3-4-5-6-7)...")
    train_ds, val_ds, test_ds, meta = create_tf_datasets(
        DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=seed, use_aug=USE_AUG)
    steps_per_epoch = meta.n_train // BATCH_SIZE
    model = build_model(INPUT_SHAPE, meta.num_classes, steps_per_epoch)

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=PATIENCE,
                      restore_best_weights=True, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'training_log.csv'), append=False),
        ModelCheckpoint(os.path.join(RESULT_DIR, 'backbone_best.keras'),
                        save_best_only=True, monitor='val_loss', verbose=1),
    ]
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        class_weight=meta.class_weight_dict,
        callbacks=callbacks,
    )

    # Learning curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, keys, title in zip(axes,
                                [('accuracy', 'val_accuracy'), ('loss', 'val_loss')],
                                ['Accuracy', 'Loss']):
        ax.plot(history.history[keys[0]], label='Train')
        ax.plot(history.history[keys[1]], label='Validation')
        ax.set_title(f'{title} — Run {run_idx+1} (seed={seed})')
        ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'backbone_learning_curve.png'), dpi=300)
    plt.show()

    # ── STEP 2: Build feature extractor & extract GAP features ─────────────
    # Uses EXTRACT_BATCH_SIZE (smaller than BATCH_SIZE) to avoid OOM during inference.
    print("\n[Step 2/3] Extracting 1792-dim GAP features from fine-tuned backbone...")
    best_backbone  = load_model(os.path.join(RESULT_DIR, 'backbone_best.keras'))
    gap_layer_name = [l.name for l in best_backbone.layers
                      if 'global_average_pooling' in l.name][0]
    feature_extractor = Model(inputs=best_backbone.input,
                               outputs=best_backbone.get_layer(gap_layer_name).output)

    X_train, y_train = extract_features(feature_extractor, DATA_DIR, 'train',
                                         meta.class_to_idx, INPUT_SHAPE, EXTRACT_BATCH_SIZE)
    X_val,   y_val   = extract_features(feature_extractor, DATA_DIR, 'val',
                                         meta.class_to_idx, INPUT_SHAPE, EXTRACT_BATCH_SIZE)
    X_test,  y_test  = extract_features(feature_extractor, DATA_DIR, 'test',
                                         meta.class_to_idx, INPUT_SHAPE, EXTRACT_BATCH_SIZE)
    print(f"  Feature shapes — train: {X_train.shape}  val: {X_val.shape}  test: {X_test.shape}")

    # ── STEP 3: Train XGBoost classifier ───────────────────────────────────
    print("\n[Step 3/3] Training XGBoost classifier on 1792-dim GAP features...")
    xgb_clf = xgb.XGBClassifier(
        n_estimators          = XGB_PARAMS['n_estimators'],
        max_depth             = XGB_PARAMS['max_depth'],
        learning_rate         = XGB_PARAMS['learning_rate'],
        subsample             = XGB_PARAMS['subsample'],
        colsample_bytree      = XGB_PARAMS['colsample_bytree'],
        min_child_weight      = XGB_PARAMS['min_child_weight'],
        gamma                 = XGB_PARAMS['gamma'],
        reg_alpha             = XGB_PARAMS['reg_alpha'],
        reg_lambda            = XGB_PARAMS['reg_lambda'],
        tree_method           = XGB_PARAMS['tree_method'],
        device                = XGB_PARAMS['device'],
        eval_metric           = XGB_PARAMS['eval_metric'],
        early_stopping_rounds = XGB_PARAMS['early_stopping_rounds'],
        random_state          = seed,
    )
    xgb_clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)

    # Save XGBoost model
    xgb_path = os.path.join(RESULT_DIR, 'xgboost_model.json')
    xgb_clf.save_model(xgb_path)

    # XGBoost feature importance plot
    fi = xgb_clf.feature_importances_
    top20_idx = np.argsort(fi)[-20:]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(range(20), fi[top20_idx], color='steelblue', edgecolor='black')
    ax.set_yticks(range(20))
    ax.set_yticklabels([f'GAP-{i}' for i in top20_idx], fontsize=8)
    ax.set_xlabel('Feature Importance (gain)', fontweight='bold')
    ax.set_title(f'XGBoost Top-20 Feature Importance — Run {run_idx+1} (seed={seed})',
                 fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'xgb_feature_importance.png'), dpi=300)
    plt.show()

    # --- Evaluation ---
    pred_probs  = xgb_clf.predict_proba(X_test)
    y_pred_run  = np.argmax(pred_probs, axis=1)
    y_true_run  = y_test
    class_names = meta.class_names

    report = classification_report(y_true_run, y_pred_run,
                                   target_names=class_names, output_dict=True, digits=4)
    with open(os.path.join(RESULT_DIR, 'classification_report.txt'), 'w') as f:
        f.write(classification_report(y_true_run, y_pred_run,
                                      target_names=class_names, digits=4))

    # Confusion matrix
    cm = confusion_matrix(y_true_run, y_pred_run)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=class_names, yticklabels=class_names, cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix — Run {run_idx+1} (seed={seed})')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'confusion_matrix.png'), dpi=300)
    plt.close()

    test_acc = np.mean(y_pred_run == y_true_run)
    all_runs_results.append({
        'run':                     run_idx + 1,
        'seed':                    seed,
        'accuracy':                test_acc,
        'precision':               report['weighted avg']['precision'],
        'recall':                  report['weighted avg']['recall'],
        'f1_score':                report['weighted avg']['f1-score'],
        'per_class_metrics':       {c: {'precision': report[c]['precision'],
                                        'recall':    report[c]['recall'],
                                        'f1-score':  report[c]['f1-score']}
                                    for c in class_names},
        'result_dir':              RESULT_DIR,
        'history':                 history.history,
        'y_true':                  y_true_run,
        'y_pred':                  y_pred_run,
        'pred_probs':              pred_probs,
        'class_names':             class_names,
        'test_filenames':          meta.test_filenames,
        'test_features':           X_test,
        'train_features':          X_train,
        'xgb_feature_importances': fi,
        'n_train':                 meta.n_train,
        'n_val':                   meta.n_val,
        'n_test':                  meta.n_test,
    })

    print(f"\n  Acc={test_acc:.4f}  P={report['weighted avg']['precision']:.4f}"
          f"  R={report['weighted avg']['recall']:.4f}"
          f"  F1={report['weighted avg']['f1-score']:.4f}")
    tf.keras.backend.clear_session()

print("\n" + "="*70)
print(f" ALL {N_RUNS} RUN(S) COMPLETED — {STRATEGY_LABEL}")
print(f" LR={LR}  |  Unfreeze={UNFREEZE_BLOCKS}  |  Aug={USE_AUG}")
print("="*70)

# Save per-strategy summary CSV
summary_path = os.path.join(BASE_RESULT_DIR, "strategy_summary.csv")
with open(summary_path, "w", newline="") as csvf:
    fieldnames = ["strategy_key", "strategy_label", "unfreeze_blocks", "lr",
                  "use_aug", "run", "seed", "accuracy", "precision", "recall", "f1_score"]
    writer = csv.DictWriter(csvf, fieldnames=fieldnames)
    writer.writeheader()
    for r in all_runs_results:
        writer.writerow({
            "strategy_key":    STRATEGY_KEY,
            "strategy_label":  STRATEGY_LABEL,
            "unfreeze_blocks": str(UNFREEZE_BLOCKS),
            "lr":              LR,
            "use_aug":         USE_AUG,
            "run":             r["run"],
            "seed":            r["seed"],
            "accuracy":        r["accuracy"],
            "precision":       r["precision"],
            "recall":          r["recall"],
            "f1_score":        r["f1_score"],
        })
print(f"Saved strategy summary -> {summary_path}")

---
## Section 2 — Results Aggregation & Scientific Reports

Aggregate metrics across all runs, generate LaTeX tables, CSV summaries, and visualizations for publication.

In [ ]:
# ========== AGGREGATE RESULTS FROM ALL RUNS ========== #
print("\n" + "="*80)
print("AGGREGATING RESULTS FROM ALL RUNS")
print("="*80 + "\n")

accuracies = [r['accuracy']  for r in all_runs_results]
precisions = [r['precision'] for r in all_runs_results]
recalls    = [r['recall']    for r in all_runs_results]
f1_scores  = [r['f1_score']  for r in all_runs_results]

overall_stats = {
    'Accuracy':  {'mean': np.mean(accuracies), 'std': np.std(accuracies), 'values': accuracies},
    'Precision': {'mean': np.mean(precisions), 'std': np.std(precisions), 'values': precisions},
    'Recall':    {'mean': np.mean(recalls),    'std': np.std(recalls),    'values': recalls},
    'F1-Score':  {'mean': np.mean(f1_scores),  'std': np.std(f1_scores),  'values': f1_scores},
}

print("OVERALL METRICS ACROSS ALL RUNS:")
print("-" * 80)
for metric_name, stats in overall_stats.items():
    print(f"{metric_name:12s}: {stats['mean']:.4f} +/- {stats['std']:.4f}")
    print(f"              Individual runs: {[f'{v:.4f}' for v in stats['values']]}")
print("-" * 80)

class_names = list(all_runs_results[0]['per_class_metrics'].keys())

per_class_stats = {}
for class_name in class_names:
    per_class_stats[class_name] = {}
    for metric in ['precision', 'recall', 'f1-score']:
        values = [r['per_class_metrics'][class_name][metric] for r in all_runs_results]
        per_class_stats[class_name][metric] = {
            'mean': np.mean(values), 'std': np.std(values), 'values': values
        }

print("\nStatistics calculated successfully.")

In [ ]:
# ========== CREATE SCIENTIFIC REPORT TABLE ========== #
print("\n" + "="*80)
print("CREATING SCIENTIFIC REPORT TABLES")
print("="*80 + "\n")

overall_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Mean':   [overall_stats[m]['mean'] for m in ['Accuracy', 'Precision', 'Recall', 'F1-Score']],
    'Std':    [overall_stats[m]['std']  for m in ['Accuracy', 'Precision', 'Recall', 'F1-Score']],
    'Run 1':  [accuracies[0], precisions[0], recalls[0], f1_scores[0]],
    'Run 2':  [accuracies[1], precisions[1], recalls[1], f1_scores[1]],
    'Run 3':  [accuracies[2], precisions[2], recalls[2], f1_scores[2]],
})
overall_df['Mean +/- Std'] = overall_df.apply(
    lambda row: f"{row['Mean']:.4f} +/- {row['Std']:.4f}", axis=1)

print("\nOVERALL PERFORMANCE METRICS (3 RUNS)")
print("="*80)
print(overall_df[['Metric', 'Mean +/- Std', 'Run 1', 'Run 2', 'Run 3']].to_string(index=False))
print("="*80)
overall_df.to_csv(os.path.join(BASE_RESULT_DIR, "overall_metrics_summary.csv"), index=False)

per_class_rows = []
for class_name in class_names:
    for metric in ['precision', 'recall', 'f1-score']:
        stats = per_class_stats[class_name][metric]
        per_class_rows.append({
            'Class': class_name, 'Metric': metric.capitalize(),
            'Mean': stats['mean'], 'Std': stats['std'],
            'Mean +/- Std': f"{stats['mean']:.4f} +/- {stats['std']:.4f}",
            'Run 1': stats['values'][0], 'Run 2': stats['values'][1], 'Run 3': stats['values'][2],
        })

per_class_df = pd.DataFrame(per_class_rows)
print("\n\nPER-CLASS PERFORMANCE METRICS (3 RUNS)")
print("="*80)
for class_name in class_names:
    class_data = per_class_df[per_class_df['Class'] == class_name]
    print(f"\n{class_name}:")
    print(class_data[['Metric', 'Mean +/- Std']].to_string(index=False))
print("="*80)
per_class_df.to_csv(os.path.join(BASE_RESULT_DIR, "per_class_metrics_summary.csv"), index=False)
print("\nSummary tables saved to CSV files.")

In [ ]:
# ========== GENERATE LATEX TABLE FOR PAPER ========== #
print("\n" + "="*80)
print("GENERATING LATEX TABLES FOR SCIENTIFIC PAPER")
print("="*80 + "\n")

latex_overall = r"""\begin{table}[h]
\centering
\caption{Overall Performance Metrics of EfficientNetB4+XGBoost (Mean $\pm$ Std over 3 runs)}
\label{tab:efficientnetb4_xgb_overall}
\begin{tabular}{lcccc}
\hline
\textbf{Metric} & \textbf{Mean $\pm$ Std} & \textbf{Run 1} & \textbf{Run 2} & \textbf{Run 3} \\
\hline
"""
for _, row in overall_df.iterrows():
    latex_overall += f"{row['Metric']} & {row['Mean +/- Std'].replace('+/-', '$\\pm$')} & {row['Run 1']:.4f} & {row['Run 2']:.4f} & {row['Run 3']:.4f} \\\\\n"
latex_overall += r"""\hline
\end{tabular}
\end{table}
"""

print("LaTeX Table - Overall Metrics:")
print(latex_overall)

latex_per_class = r"""\begin{table}[h]
\centering
\caption{Per-Class Performance Metrics of EfficientNetB4+XGBoost (Mean $\pm$ Std over 3 runs)}
\label{tab:efficientnetb4_xgb_per_class}
\begin{tabular}{lccc}
\hline
\textbf{Class} & \textbf{Precision} & \textbf{Recall} & \textbf{F1-Score} \\
\hline
"""
for class_name in class_names:
    prec = per_class_stats[class_name]['precision']
    rec  = per_class_stats[class_name]['recall']
    f1   = per_class_stats[class_name]['f1-score']
    latex_per_class += (f"{class_name} & {prec['mean']:.4f} $\\pm$ {prec['std']:.4f} & "
                        f"{rec['mean']:.4f} $\\pm$ {rec['std']:.4f} & "
                        f"{f1['mean']:.4f} $\\pm$ {f1['std']:.4f} \\\\\n")
latex_per_class += r"""\hline
\end{tabular}
\end{table}
"""
print("\n" + "="*80)
print("LaTeX Table - Per-Class Metrics:")
print(latex_per_class)

with open(os.path.join(BASE_RESULT_DIR, "latex_tables.tex"), "w") as f:
    f.write("% Overall Metrics Table\n")
    f.write(latex_overall)
    f.write("\n\n% Per-Class Metrics Table\n")
    f.write(latex_per_class)

print("\nLaTeX tables saved to 'latex_tables.tex'.")

In [ ]:
# ========== VISUALIZATION OF RESULTS ACROSS RUNS ========== #
print("\n" + "="*80)
print("CREATING VISUALIZATIONS")
print("="*80 + "\n")

metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
means = [overall_stats[m]['mean'] for m in metrics_list]
stds  = [overall_stats[m]['std']  for m in metrics_list]

# 1. Bar plot with error bars
fig, ax = plt.subplots(figsize=(10, 6))
x_pos = np.arange(len(metrics_list))
ax.bar(x_pos, means, yerr=stds, capsize=10, alpha=0.8, color='steelblue', edgecolor='black')
ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('EfficientNetB4 + XGBoost Performance (Mean +/- Std over 3 runs)', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos); ax.set_xticklabels(metrics_list)
ax.set_ylim([0, 1.05]); ax.grid(axis='y', alpha=0.3)
for i, (mean, std) in enumerate(zip(means, stds)):
    ax.text(i, mean + std + 0.02, f'{mean:.4f}\n+/-{std:.4f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, "overall_metrics_barplot.png"), dpi=300, bbox_inches='tight')
plt.show()

# 2. Box plot
fig, ax = plt.subplots(figsize=(10, 6))
data_for_box = [accuracies, precisions, recalls, f1_scores]
bp = ax.boxplot(data_for_box, labels=metrics_list, patch_artist=True, showmeans=True,
                meanprops=dict(marker='D', markerfacecolor='red', markersize=8))
for patch in bp['boxes']:
    patch.set_facecolor('lightblue'); patch.set_alpha(0.7)
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Metrics Across 3 Runs', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1.05]); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, "metrics_boxplot.png"), dpi=300, bbox_inches='tight')
plt.show()

# 3. Per-class F1-Score
fig, ax = plt.subplots(figsize=(12, 6))
class_f1_means = [per_class_stats[c]['f1-score']['mean'] for c in class_names]
class_f1_stds  = [per_class_stats[c]['f1-score']['std']  for c in class_names]
x_pos = np.arange(len(class_names))
ax.bar(x_pos, class_f1_means, yerr=class_f1_stds, capsize=5,
       alpha=0.8, color='coral', edgecolor='black')
ax.set_xlabel('Class', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax.set_title('Per-Class F1-Score (Mean +/- Std over 3 runs)', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos); ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_ylim([0, 1.05]); ax.grid(axis='y', alpha=0.3)
for i, (mean, std) in enumerate(zip(class_f1_means, class_f1_stds)):
    ax.text(i, mean + std + 0.02, f'{mean:.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, "per_class_f1score.png"), dpi=300, bbox_inches='tight')
plt.show()

print("All visualizations created and saved.")

In [ ]:
# ========== GENERATE COMPREHENSIVE SUMMARY REPORT ========== #
print("\n" + "="*80)
print("GENERATING COMPREHENSIVE SUMMARY REPORT")
print("="*80 + "\n")

report_lines = []
report_lines.append("="*100)
report_lines.append("EfficientNetB4 + XGBoost - MULTI-RUN EXPERIMENT REPORT")
report_lines.append("="*100)
report_lines.append(f"\nGenerated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
report_lines.append(f"Strategy  : {STRATEGY_LABEL}")
report_lines.append(f"Seeds     : {RANDOM_SEEDS}")
report_lines.append(f"Number of Runs: {len(RANDOM_SEEDS)}")

report_lines.append("\n" + "="*100)
report_lines.append("OVERALL PERFORMANCE METRICS (Mean +/- Std)")
report_lines.append("="*100)
report_lines.append(f"{'Metric':<20} {'Mean +/- Std':<25} {'Run 1':<15} {'Run 2':<15} {'Run 3':<15}")
report_lines.append("-"*100)
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    stats = overall_stats[metric]
    report_lines.append(f"{metric:<20} {stats['mean']:.4f} +/- {stats['std']:.4f}      "
                        f"{stats['values'][0]:<15.4f} {stats['values'][1]:<15.4f} {stats['values'][2]:<15.4f}")

report_lines.append("\n" + "="*100)
report_lines.append("PER-CLASS PERFORMANCE METRICS (Mean +/- Std)")
report_lines.append("="*100)
for class_name in class_names:
    report_lines.append(f"\nClass: {class_name}")
    report_lines.append("-"*100)
    report_lines.append(f"{'Metric':<20} {'Mean +/- Std':<25} {'Run 1':<15} {'Run 2':<15} {'Run 3':<15}")
    report_lines.append("-"*100)
    for metric in ['precision', 'recall', 'f1-score']:
        stats = per_class_stats[class_name][metric]
        report_lines.append(f"{metric.capitalize():<20} {stats['mean']:.4f} +/- {stats['std']:.4f}      "
                            f"{stats['values'][0]:<15.4f} {stats['values'][1]:<15.4f} {stats['values'][2]:<15.4f}")

report_lines.append("\n" + "="*100)
report_lines.append("INDIVIDUAL RUN DETAILS")
report_lines.append("="*100)
for run_result in all_runs_results:
    report_lines.append(f"\nRun {run_result['run']} (Seed: {run_result['seed']})")
    report_lines.append(f"  Accuracy:  {run_result['accuracy']:.4f}")
    report_lines.append(f"  Precision: {run_result['precision']:.4f}")
    report_lines.append(f"  Recall:    {run_result['recall']:.4f}")
    report_lines.append(f"  F1-Score:  {run_result['f1_score']:.4f}")
    report_lines.append(f"  Result Dir: {run_result['result_dir']}")

best_run_idx  = np.argmax(accuracies)
worst_run_idx = np.argmin(accuracies)
report_lines.append(f"\nBest  Run (by Accuracy): Run {best_run_idx+1} (Seed {RANDOM_SEEDS[best_run_idx]}) -> {accuracies[best_run_idx]:.4f}")
report_lines.append(f"Worst Run (by Accuracy): Run {worst_run_idx+1} (Seed {RANDOM_SEEDS[worst_run_idx]}) -> {accuracies[worst_run_idx]:.4f}")
report_lines.append("\nVariability (Coefficient of Variation):")
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    cv = (overall_stats[metric]['std'] / overall_stats[metric]['mean']) * 100
    report_lines.append(f"  {metric}: {cv:.2f}%")

report_lines.append("\n" + "="*100)
report_lines.append("END OF REPORT")
report_lines.append("="*100)

report_text = "\n".join(report_lines)
print(report_text)

with open(os.path.join(BASE_RESULT_DIR, "MULTI_RUN_SUMMARY_REPORT.txt"), "w", encoding="utf-8") as f:
    f.write(report_text)

print("\nComprehensive summary report saved to 'MULTI_RUN_SUMMARY_REPORT.txt'.")

In [ ]:
# ========== ZIP ALL RESULTS ========== #
import shutil

zip_output_path = "/kaggle/working/EfficientNetB4_XGBoost_MultiRun_Complete"
print(f"Source: {BASE_RESULT_DIR}")
print(f"Output: {zip_output_path}.zip")

shutil.make_archive(zip_output_path, 'zip', BASE_RESULT_DIR)

zip_size = os.path.getsize(f"{zip_output_path}.zip") / (1024*1024)
print(f"\nComplete multi-run report archived successfully.")
print(f"Archive size: {zip_size:.2f} MB")
print(f"Location    : {zip_output_path}.zip")
print("\n" + "="*80)
print("ALL DONE. Ready for scientific paper submission.")
print("="*80)

---
## Section 3 — Advanced Analysis for Thesis / Paper

Reports below aggregate data from **all runs** (dataset, confusion matrix, ROC curves, Kappa/MCC) and then provide **single-run deep-dive** analysis (Grad-CAM, t-SNE, error analysis, XGBoost feature importance).

> **To analyse a specific run:** change `SELECTED_RUN` in the *Select Run* cell below (1 / 2 / 3).

In [ ]:
# ========== DATASET DISTRIBUTION ANALYSIS ========== #
splits = ['train', 'val', 'test']
split_counts = {}
for split in splits:
    split_dir = os.path.join(DATA_DIR, split)
    counts = {cls: len(os.listdir(os.path.join(split_dir, cls)))
              for cls in sorted(os.listdir(split_dir))
              if os.path.isdir(os.path.join(split_dir, cls))}
    split_counts[split] = counts

dist_df = pd.DataFrame(split_counts).fillna(0).astype(int)
dist_df['Total'] = dist_df.sum(axis=1)
print("Dataset Distribution:")
print(dist_df.to_string())
print(f"\nTotal images : {dist_df['Total'].sum()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dist_df[splits].plot(kind='bar', ax=axes[0],
                     color=['steelblue', 'coral', 'seagreen'],
                     edgecolor='black', width=0.7)
axes[0].set_title('Sample Count per Class per Split', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Class'); axes[0].set_ylabel('Number of Images')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].legend(title='Split'); axes[0].grid(axis='y', alpha=0.3)
for container in axes[0].containers:
    axes[0].bar_label(container, fontsize=7, padding=1)

axes[1].pie(dist_df['Total'], labels=dist_df.index, autopct='%1.1f%%',
            startangle=140, colors=plt.cm.Set3.colors[:len(dist_df)])
axes[1].set_title('Overall Class Distribution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'dataset_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()
dist_df.to_csv(os.path.join(BASE_RESULT_DIR, 'dataset_distribution.csv'))
print("Saved → dataset_distribution.png, dataset_distribution.csv")

In [ ]:
# ========== NORMALIZED CONFUSION MATRIX (Aggregate across runs) ========== #
cls    = all_runs_results[0]['class_names']
n_cls  = len(cls)
agg_cm = np.zeros((n_cls, n_cls))
for r in all_runs_results:
    agg_cm += confusion_matrix(r['y_true'], r['y_pred']).astype(float)

agg_cm_norm = agg_cm / agg_cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(agg_cm.astype(int), annot=True, fmt='d',
            xticklabels=cls, yticklabels=cls, cmap='Blues', ax=axes[0])
axes[0].set_title('Aggregate Confusion Matrix (sum, 3 runs)', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(agg_cm_norm, annot=True, fmt='.2%',
            xticklabels=cls, yticklabels=cls, cmap='Blues', ax=axes[1])
axes[1].set_title('Normalized Confusion Matrix (row = recall)', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.suptitle('Confusion Matrices — EfficientNetB4 + XGBoost  (Aggregate over 3 runs)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'aggregate_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Saved → aggregate_confusion_matrix.png")

print("\nPer-class recall (diagonal of normalized CM):")
for i, cname in enumerate(cls):
    print(f"  {cname:<25} {agg_cm_norm[i, i]:.4f}")

In [ ]:
# ========== MULTI-CLASS ROC CURVES + AUC (One-vs-Rest, Aggregate) ========== #
all_y_true = np.concatenate([r['y_true']     for r in all_runs_results])
all_probs  = np.concatenate([r['pred_probs'] for r in all_runs_results])
cls        = all_runs_results[0]['class_names']
n_cls      = len(cls)
y_bin      = label_binarize(all_y_true, classes=range(n_cls))

fig, axes  = plt.subplots(1, 2, figsize=(14, 6))
tab_colors = plt.cm.tab10.colors
auc_scores = {}

fpr_d, tpr_d = {}, {}
for i, cname in enumerate(cls):
    fpr_d[i], tpr_d[i], _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr_d[i], tpr_d[i])
    auc_scores[cname] = roc_auc
    axes[0].plot(fpr_d[i], tpr_d[i], color=tab_colors[i], lw=2,
                 label=f'{cname}  (AUC={roc_auc:.4f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set(xlim=[0, 1], ylim=[0, 1.01],
            xlabel='False Positive Rate', ylabel='True Positive Rate',
            title='Per-Class ROC Curves (OvR) — Aggregate')
axes[0].legend(loc='lower right', fontsize=8); axes[0].grid(alpha=0.3)

all_fpr  = np.unique(np.concatenate([fpr_d[i] for i in range(n_cls)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_cls):
    mean_tpr += np.interp(all_fpr, fpr_d[i], tpr_d[i])
mean_tpr /= n_cls
macro_auc = auc(all_fpr, mean_tpr)

axes[1].plot(all_fpr, mean_tpr, 'b-', lw=2, label=f'Macro-avg  (AUC={macro_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set(xlim=[0, 1], ylim=[0, 1.01],
            xlabel='False Positive Rate', ylabel='True Positive Rate',
            title='Macro-Average ROC Curve')
axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)

plt.suptitle('ROC Curves — EfficientNetB4 + XGBoost  (Aggregate over all runs)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'roc_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

print("\nAUC Scores (aggregate):")
print("-" * 40)
for cname, av in auc_scores.items():
    print(f"  {cname:<25} {av:.4f}")
weights      = [np.sum(all_y_true == i) for i in range(n_cls)]
weighted_auc = np.average(list(auc_scores.values()), weights=weights)
print(f"\n  Macro-average AUC   : {macro_auc:.4f}")
print(f"  Weighted-avg AUC    : {weighted_auc:.4f}")

auc_df = pd.DataFrame({'Class': list(auc_scores.keys()), 'AUC': list(auc_scores.values())})
auc_df = pd.concat([auc_df,
                    pd.DataFrame([{'Class': 'Macro-Average',    'AUC': macro_auc},
                                  {'Class': 'Weighted-Average', 'AUC': weighted_auc}])],
                   ignore_index=True)
auc_df.to_csv(os.path.join(BASE_RESULT_DIR, 'auc_scores.csv'), index=False)
print("Saved → roc_curves.png, auc_scores.csv")

In [ ]:
# ========== ADDITIONAL CLASSIFICATION METRICS (Kappa, MCC, Balanced Acc) ========== #
print("="*70)
print("ADDITIONAL METRICS — ALL RUNS")
print("="*70)

extra_rows = []
for r in all_runs_results:
    kappa   = cohen_kappa_score(r['y_true'], r['y_pred'])
    mcc     = matthews_corrcoef(r['y_true'], r['y_pred'])
    bal_acc = balanced_accuracy_score(r['y_true'], r['y_pred'])
    extra_rows.append({
        'Run': r['run'], 'Seed': r['seed'],
        'Accuracy': r['accuracy'], 'Balanced Accuracy': bal_acc,
        "Cohen's Kappa": kappa, 'MCC': mcc, 'F1-Score (w)': r['f1_score'],
    })
    print(f"\nRun {r['run']} (seed={r['seed']}):")
    print(f"  Accuracy          : {r['accuracy']:.4f}")
    print(f"  Balanced Accuracy : {bal_acc:.4f}")
    print(f"  Cohen's Kappa     : {kappa:.4f}")
    print(f"  MCC               : {mcc:.4f}")
    print(f"  F1-Score (w-avg)  : {r['f1_score']:.4f}")

extra_df  = pd.DataFrame(extra_rows)
num_cols  = ['Accuracy', 'Balanced Accuracy', "Cohen's Kappa", 'MCC', 'F1-Score (w)']
mean_row  = {'Run': 'Mean', 'Seed': '—', **{c: extra_df[c].mean() for c in num_cols}}
std_row   = {'Run': 'Std',  'Seed': '—', **{c: extra_df[c].std()  for c in num_cols}}
summary_extra = pd.concat([extra_df, pd.DataFrame([mean_row, std_row])], ignore_index=True)

print("\n" + "="*70)
print("SUMMARY TABLE")
print(summary_extra.to_string(index=False))

extra_df.to_csv(os.path.join(BASE_RESULT_DIR, 'additional_metrics.csv'), index=False)
summary_extra.to_csv(os.path.join(BASE_RESULT_DIR, 'additional_metrics_summary.csv'), index=False)

print("\nInterpretation:")
print("  Cohen's Kappa > 0.80 → Almost perfect agreement (Landis & Koch)")
print("  MCC ∈ [−1, 1]; values close to 1 indicate best classification")
print("  Balanced Accuracy corrects for class imbalance")
print("Saved → additional_metrics.csv, additional_metrics_summary.csv")

In [ ]:
# ========== TRAINING CONVERGENCE ANALYSIS ========== #
colors3 = ['steelblue', 'coral', 'seagreen']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for r, c in zip(all_runs_results, colors3):
    axes[0, 0].plot(r['history']['val_accuracy'], color=c,
                    label=f"Run {r['run']} (seed={r['seed']})", lw=2)
axes[0, 0].set_title('Validation Accuracy — All Runs', fontweight='bold')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

for r, c in zip(all_runs_results, colors3):
    axes[0, 1].plot(r['history']['val_loss'], color=c,
                    label=f"Run {r['run']} (seed={r['seed']})", lw=2)
axes[0, 1].set_title('Validation Loss — All Runs', fontweight='bold')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

best_epochs   = [np.argmin(r['history']['val_loss']) + 1 for r in all_runs_results]
run_labels    = [f"Run {r['run']}\n(seed={r['seed']})" for r in all_runs_results]
best_val_accs = [r['history']['val_accuracy'][np.argmin(r['history']['val_loss'])]
                 for r in all_runs_results]

bars = axes[1, 0].bar(run_labels, best_epochs, color=colors3[:len(all_runs_results)],
                      edgecolor='black', alpha=0.85)
axes[1, 0].set_title('Best Epoch per Run (EarlyStopping)', fontweight='bold')
axes[1, 0].set_ylabel('Epoch'); axes[1, 0].grid(axis='y', alpha=0.3)
for bar, ep in zip(bars, best_epochs):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    str(ep), ha='center', fontweight='bold')

for r, c in zip(all_runs_results, colors3):
    gap = np.array(r['history']['accuracy']) - np.array(r['history']['val_accuracy'])
    axes[1, 1].plot(gap, color=c, label=f"Run {r['run']}", lw=2)
axes[1, 1].axhline(0, color='black', linestyle='--', lw=1)
axes[1, 1].set_title('Train−Val Accuracy Gap (Overfitting Indicator)', fontweight='bold')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Train Acc − Val Acc')
axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

plt.suptitle('Training Convergence Analysis — EfficientNetB4 + XGBoost',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'convergence_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()

print("Best epochs    :", {f"Run {r['run']}": be for r, be in zip(all_runs_results, best_epochs)})
print("Val acc @ best :", {f"Run {r['run']}": f"{a:.4f}" for r, a in zip(all_runs_results, best_val_accs)})
print("Saved → convergence_analysis.png")

In [ ]:
# ========== AGGREGATE XGBoost FEATURE IMPORTANCE ========== #
print("="*70)
print("AGGREGATE XGBoost FEATURE IMPORTANCE (mean across 3 runs)")
print("="*70)

fi_all  = np.stack([r['xgb_feature_importances'] for r in all_runs_results])
fi_mean = fi_all.mean(axis=0)
fi_std  = fi_all.std(axis=0)

top20_idx = np.argsort(fi_mean)[-20:]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(range(20), fi_mean[top20_idx], xerr=fi_std[top20_idx],
             color='steelblue', edgecolor='black', capsize=4)
axes[0].set_yticks(range(20))
axes[0].set_yticklabels([f'GAP-{i}' for i in top20_idx], fontsize=8)
axes[0].set_xlabel('Mean Importance (gain) ± Std', fontweight='bold')
axes[0].set_title('Top-20 GAP Dimensions — Mean ± Std (3 runs)', fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

top10_idx = top20_idx[-10:]
x  = np.arange(10)
bw = 0.25
for run_i, r in enumerate(all_runs_results):
    axes[1].bar(x + run_i * bw, r['xgb_feature_importances'][top10_idx],
                width=bw, label=f"Run {r['run']}", alpha=0.8, edgecolor='black')
axes[1].set_xticks(x + bw)
axes[1].set_xticklabels([f'GAP-{i}' for i in top10_idx], rotation=30, ha='right', fontsize=8)
axes[1].set_ylabel('Feature Importance', fontweight='bold')
axes[1].set_title('Top-10 GAP Dimensions — Per-Run Comparison', fontweight='bold')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('XGBoost Feature Importance — 1792-dim GAP Features',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'xgb_feature_importance_aggregate.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"Feature dim : {len(fi_mean)}")
print(f"Top-3 most important GAP dimensions: {top20_idx[-3:][::-1].tolist()}")
print("Saved → xgb_feature_importance_aggregate.png")

In [ ]:
# ========== SELECT RUN FOR SINGLE-RUN ANALYSIS ========== #
# Change SELECTED_RUN to 1, 2, or 3 to analyse a different experiment.
SELECTED_RUN = len(RANDOM_SEEDS)   # default: last run

run_data      = all_runs_results[SELECTED_RUN - 1]
RESULT_DIR    = run_data['result_dir']
y_true        = run_data['y_true']
y_pred        = run_data['y_pred']
pred_probs    = run_data['pred_probs']
class_names   = run_data['class_names']
test_acc      = run_data['accuracy']
test_features = run_data['test_features']   # 1792-dim GAP features (pre-extracted)

n_train        = run_data['n_train']
n_val          = run_data['n_val']
n_test         = run_data['n_test']
test_filenames = run_data['test_filenames']
test_dir       = os.path.join(DATA_DIR, 'test')

history = SimpleNamespace(history=run_data['history'])

# Reload best backbone model (for Grad-CAM)
backbone_model = load_model(os.path.join(RESULT_DIR, 'backbone_best.keras'))

# Rebuild feature extractor from GAP layer
gap_layer_name    = [l.name for l in backbone_model.layers if 'global_average_pooling' in l.name][0]
feature_extractor = Model(inputs=backbone_model.input,
                           outputs=backbone_model.get_layer(gap_layer_name).output)

# Reload XGBoost model
xgb_clf = xgb.XGBClassifier()
xgb_clf.load_model(os.path.join(RESULT_DIR, 'xgboost_model.json'))

# Use backbone_model as 'model' for Grad-CAM cells below
model = backbone_model

# Rebuild test tf.data pipeline for inference-speed measurement
_, _, test_ds, _ = create_tf_datasets(DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=run_data['seed'])

print(f"Analysing Run {SELECTED_RUN}  (seed={run_data['seed']})")
print(f"  Classes       : {class_names}")
print(f"  Train/Val/Test: {n_train}/{n_val}/{n_test}")
print(f"  Accuracy      : {run_data['accuracy']:.4f}")
print(f"  F1-Score      : {run_data['f1_score']:.4f}")
print(f"  Feature dim   : {test_features.shape[1]}")
print(f"  Dir           : {RESULT_DIR}")

In [ ]:
# ========== GRAD-CAM VISUALIZATION ========== #

def make_gradcam_heatmap(img_array, grad_model, pred_index=None):
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    grads  = tape.gradient(class_channel, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_heatmap(orig_img_array, heatmap, alpha=0.4):
    heatmap_uint8 = np.uint8(255 * heatmap)
    jet_colors    = plt.cm.jet(np.arange(256))[:, :3]
    jet_heatmap   = jet_colors[heatmap_uint8]
    jet_heatmap   = tf.keras.preprocessing.image.array_to_img(jet_heatmap)
    jet_heatmap   = jet_heatmap.resize((orig_img_array.shape[1], orig_img_array.shape[0]))
    jet_heatmap   = img_to_array(jet_heatmap)
    superimposed  = jet_heatmap * alpha + orig_img_array
    return np.clip(superimposed / superimposed.max(), 0, 1)

last_conv_name = [l.name for l in model.layers
                  if isinstance(l, tf.keras.layers.Conv2D)][-1]
print(f"Last conv layer: {last_conv_name}")
grad_model = Model(inputs=model.inputs,
                   outputs=[model.get_layer(last_conv_name).output, model.output])

n_cls = len(class_names)
fig, axes = plt.subplots(n_cls, 3, figsize=(12, 4 * n_cls))
if n_cls == 1:
    axes = axes[np.newaxis, :]

for ci, cname in enumerate(class_names):
    ok_idx = np.where((y_true == ci) & (y_pred == ci))[0]
    idx    = ok_idx[0] if len(ok_idx) > 0 else np.where(y_true == ci)[0][0]
    fpath  = os.path.join(test_dir, test_filenames[idx])

    img_orig = load_img(fpath, target_size=INPUT_SHAPE[:2])
    img_arr  = img_to_array(img_orig)
    img_proc = efficientnet_preprocess(np.expand_dims(img_arr.copy(), 0))

    heatmap = make_gradcam_heatmap(img_proc, grad_model, pred_index=ci)
    overlay = overlay_heatmap(img_arr, heatmap)

    axes[ci, 0].imshow(img_orig); axes[ci, 0].axis('off')
    axes[ci, 0].set_title(f'Original\nClass: {cname}', fontsize=9)
    axes[ci, 1].imshow(heatmap, cmap='jet'); axes[ci, 1].axis('off')
    axes[ci, 1].set_title('Grad-CAM Heatmap', fontsize=9)
    axes[ci, 2].imshow(overlay); axes[ci, 2].axis('off')
    conf = pred_probs[idx][y_pred[idx]]
    axes[ci, 2].set_title(f'Overlay\nConf={conf:.2%}', fontsize=9)

plt.suptitle(f'Grad-CAM — EfficientNetB4 Backbone  (Run {SELECTED_RUN})',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'gradcam_visualization.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved → gradcam_visualization.png")

In [ ]:
# ========== t-SNE FEATURE EMBEDDING VISUALIZATION ========== #
# Uses pre-extracted 1792-dim GAP features — no re-inference needed.
print("Computing t-SNE embedding from pre-extracted 1792-dim GAP features...")
tsne        = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
features_2d = tsne.fit_transform(test_features)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
palette   = plt.cm.tab10.colors

# Left: color by true class
for ci, cname in enumerate(class_names):
    mask = y_true == ci
    axes[0].scatter(features_2d[mask, 0], features_2d[mask, 1],
                    c=[palette[ci]], label=cname, alpha=0.7, s=20)
axes[0].set_title(f't-SNE — True Labels  (Run {SELECTED_RUN})', fontweight='bold')
axes[0].legend(markerscale=2, fontsize=9); axes[0].grid(alpha=0.3)
axes[0].set_xlabel('Dim 1'); axes[0].set_ylabel('Dim 2')

# Right: correct vs misclassified
cm_mask = y_true == y_pred
axes[1].scatter(features_2d[cm_mask, 0], features_2d[cm_mask, 1],
                c='steelblue', label=f'Correct ({cm_mask.sum()})', alpha=0.6, s=20)
axes[1].scatter(features_2d[~cm_mask, 0], features_2d[~cm_mask, 1],
                c='red', label=f'Misclassified ({(~cm_mask).sum()})',
                alpha=0.9, s=50, marker='x')
axes[1].set_title('t-SNE — Correct vs Misclassified', fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
axes[1].set_xlabel('Dim 1'); axes[1].set_ylabel('Dim 2')

plt.suptitle(f't-SNE (1792-dim GAP) — EfficientNetB4 + XGBoost  (Run {SELECTED_RUN})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'tsne_embedding.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Saved → tsne_embedding.png")

In [ ]:
# ========== ERROR ANALYSIS ========== #
print("="*70)
print("ERROR ANALYSIS")
print("="*70)

confusion_counts = defaultdict(int)
for yt, yp in zip(y_true, y_pred):
    if yt != yp:
        confusion_counts[(class_names[yt], class_names[yp])] += 1

print(f"\nTotal misclassifications: {sum(confusion_counts.values())} / {len(y_true)}")
print(f"Overall accuracy: {np.mean(y_true == y_pred):.4f}\n")
print("Most common confused pairs (True → Predicted):")
for (tc, pc), cnt in sorted(confusion_counts.items(), key=lambda x: -x[1])[:10]:
    pct = cnt / np.sum(y_true == class_names.index(tc)) * 100
    print(f"  {tc:<22} → {pc:<22}  {cnt:3d} samples  ({pct:.1f}% of class)")

correct_mask = y_true == y_pred
correct_conf = pred_probs[correct_mask][np.arange(correct_mask.sum()), y_pred[correct_mask]]
wrong_conf   = pred_probs[~correct_mask][np.arange((~correct_mask).sum()), y_pred[~correct_mask]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(correct_conf, bins=20, color='steelblue', alpha=0.7, edgecolor='black',
             label=f'Correct (n={len(correct_conf)})')
axes[0].hist(wrong_conf, bins=20, color='salmon', alpha=0.7, edgecolor='black',
             label=f'Wrong (n={len(wrong_conf)})')
axes[0].axvline(0.5, color='black', linestyle='--', lw=1)
axes[0].set_title('XGBoost Prediction Confidence Distribution', fontweight='bold')
axes[0].set_xlabel('Confidence (predict_proba)'); axes[0].set_ylabel('Count')
axes[0].legend(); axes[0].grid(alpha=0.3)

class_acc  = [(cn, np.mean(y_pred[y_true == ci] == ci), (y_true == ci).sum())
              for ci, cn in enumerate(class_names)]
class_acc.sort(key=lambda x: x[1])
names, accs, counts = zip(*class_acc)
colors_bar = plt.cm.RdYlGn(np.array(accs))
bars = axes[1].barh(names, accs, color=colors_bar, edgecolor='black', height=0.6)
axes[1].set_title('Per-Class Accuracy (sorted)', fontweight='bold')
axes[1].set_xlabel('Accuracy'); axes[1].set_xlim([0, 1.15])
axes[1].grid(axis='x', alpha=0.3)
for bar, acc, n in zip(bars, accs, counts):
    axes[1].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{acc:.3f}  (n={n})', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'error_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()

wrong_idx       = np.where(y_true != y_pred)[0]
wrong_conf_vals = pred_probs[wrong_idx][np.arange(len(wrong_idx)), y_pred[wrong_idx]]
top_wrong       = wrong_idx[np.argsort(-wrong_conf_vals)[:10]]
print("\nTop-10 high-confidence incorrect predictions:")
for i, idx in enumerate(top_wrong):
    fname = test_filenames[idx] if idx < len(test_filenames) else str(idx)
    print(f"  {i+1:2d}. True={class_names[y_true[idx]]:<20}  "
          f"Pred={class_names[y_pred[idx]]:<20}  "
          f"Conf={pred_probs[idx][y_pred[idx]]:.4f}  ({fname})")
print("\nSaved → error_analysis.png")

In [ ]:
# ========== TOP-5 MISCLASSIFIED IMAGES PER CONFUSION PAIR ========== #
import gc

TOP_N         = 5
GRADCAM_ALPHA = 0.45

wrong_indices = np.where(y_true != y_pred)[0]

last_conv_name = [l.name for l in model.layers
                  if isinstance(l, tf.keras.layers.Conv2D)][-1]
grad_model_vis = tf.keras.Model(
    inputs  = model.inputs,
    outputs = [model.get_layer(last_conv_name).output, model.output]
)

def _gradcam(img_path, target_class):
    try:
        img_orig = load_img(img_path, target_size=INPUT_SHAPE[:2])
        img_arr  = img_to_array(img_orig)
        img_proc = efficientnet_preprocess(tf.cast(np.expand_dims(img_arr.copy(), 0), tf.float32))
        with tf.GradientTape() as tape:
            conv_out, preds = grad_model_vis(img_proc, training=False)
            class_score     = preds[:, target_class]
        grads   = tape.gradient(class_score, conv_out)
        pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
        heatmap = tf.squeeze(conv_out[0] @ pooled[..., tf.newaxis])
        heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
        heatmap = heatmap.numpy()
        h_uint8    = np.uint8(255 * heatmap)
        jet_colors = plt.cm.jet(np.arange(256))[:, :3]
        jet_hm_img = tf.keras.preprocessing.image.array_to_img(jet_colors[h_uint8])
        jet_hm_img = jet_hm_img.resize((img_arr.shape[1], img_arr.shape[0]))
        jet_hm_arr = img_to_array(jet_hm_img)
        overlay    = jet_hm_arr * GRADCAM_ALPHA + img_arr
        overlay    = np.clip(overlay / overlay.max(), 0, 1)
        conf       = float(preds.numpy()[0, target_class])
        return np.clip(img_arr / 255.0, 0, 1), heatmap, overlay, conf
    except Exception as e:
        print(f'  [WARN] Grad-CAM failed: {type(e).__name__}: {str(e)[:80]}')
        try:
            orig_float = np.clip(img_to_array(load_img(img_path, target_size=INPUT_SHAPE[:2])) / 255.0, 0, 1)
        except Exception:
            orig_float = np.zeros((*INPUT_SHAPE[:2], 3))
        return orig_float, None, orig_float, 0.0

pair_dict = defaultdict(list)
for idx in wrong_indices:
    pair = (int(y_true[idx]), int(y_pred[idx]))
    pair_dict[pair].append((idx, float(pred_probs[idx][y_pred[idx]])))

sorted_pairs = sorted(pair_dict.items(), key=lambda x: -len(x[1]))
print(f'Found {len(sorted_pairs)} unique (True -> Predicted) confusion pairs')
print(f'Generating Top-{TOP_N} misclassified image panels...\n')

report_rows = []

for (ti, pi), samples in sorted_pairs:
    true_name   = class_names[ti]
    pred_name   = class_names[pi]
    n_total     = len(samples)
    top_samples = sorted(samples, key=lambda x: -x[1])[:TOP_N]
    n_show      = len(top_samples)

    fig, axes = plt.subplots(n_show, 3, figsize=(12, max(3.5, n_show * 3.5)))
    if n_show == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(f'True: {true_name}  ->  Predicted: {pred_name}  ({n_total} total wrong)',
                 fontsize=13, fontweight='bold', color='crimson', y=1.01)
    for col, title in enumerate(['Original Image', 'Grad-CAM Heatmap', 'Overlay']):
        axes[0, col].set_title(title, fontsize=10, fontweight='bold', pad=6)

    for row, (idx, _) in enumerate(top_samples):
        fpath     = os.path.join(test_dir, test_filenames[idx])
        true_conf = float(pred_probs[idx][ti])
        pred_conf = float(pred_probs[idx][pi])
        orig, heatmap, overlay, _ = _gradcam(fpath, pi)

        axes[row, 0].imshow(orig); axes[row, 0].axis('off')
        axes[row, 0].text(0.02, 0.02, f'True: {true_name}\n(p={true_conf:.2%})',
            transform=axes[row, 0].transAxes, fontsize=8, color='lime',
            fontweight='bold', bbox=dict(facecolor='black', alpha=0.55, pad=2))

        if heatmap is not None:
            axes[row, 1].imshow(heatmap, cmap='jet')
            axes[row, 1].text(0.02, 0.02, f'Layer:\n{last_conv_name}',
                transform=axes[row, 1].transAxes, fontsize=7, color='white',
                bbox=dict(facecolor='black', alpha=0.55, pad=2))
        else:
            axes[row, 1].imshow(orig)
            axes[row, 1].text(0.02, 0.02, 'Grad-CAM\nunavailable\n(OOM)',
                transform=axes[row, 1].transAxes, fontsize=8, color='red',
                fontweight='bold', bbox=dict(facecolor='black', alpha=0.60, pad=2))
        axes[row, 1].axis('off')

        top3_idx = np.argsort(-pred_probs[idx])[:3]
        top3_str = '\n'.join([f'  {class_names[k]}: {pred_probs[idx][k]:.2%}' for k in top3_idx])
        axes[row, 2].imshow(overlay); axes[row, 2].axis('off')
        axes[row, 2].text(0.02, 0.02,
            f'WRONG: {pred_name}\n(p={pred_conf:.2%})\n\nTop-3:\n{top3_str}',
            transform=axes[row, 2].transAxes, fontsize=7.5,
            color='yellow', fontweight='bold', bbox=dict(facecolor='black', alpha=0.60, pad=3))

        report_rows.append({
            'true_class': true_name, 'pred_class': pred_name, 'sample': row + 1,
            'filename': test_filenames[idx], 'true_conf': round(true_conf, 4),
            'pred_conf': round(pred_conf, 4), 'gradcam_ok': heatmap is not None,
            'total_in_pair': n_total,
        })

    plt.tight_layout()
    safe_fname = f'top{TOP_N}_wrong_{true_name}_as_{pred_name}.png'
    plt.savefig(os.path.join(RESULT_DIR, safe_fname), dpi=200, bbox_inches='tight')
    plt.show()
    print(f'  [{true_name} -> {pred_name}]  {n_total} errors — saved {safe_fname}')
    gc.collect()

wrong_report_df = pd.DataFrame(report_rows)
csv_path = os.path.join(RESULT_DIR, 'top5_misclassified_report.csv')
wrong_report_df.to_csv(csv_path, index=False)
print(f'\nCSV saved ({len(wrong_report_df)} rows) -> top5_misclassified_report.csv')
print('Done.')

In [ ]:
# ========== PREPARE PATHS ========== #
filepaths = [os.path.join(test_dir, f) for f in test_filenames]

In [ ]:
# ========== FIND CONFUSION TYPES ========== #
from collections import defaultdict

confusion_dict = defaultdict(list)
for i in range(len(y_true)):
    if y_true[i] != y_pred[i]:
        key        = (class_names[y_true[i]], class_names[y_pred[i]])
        confidence = pred_probs[i][y_pred[i]]
        confusion_dict[key].append((filepaths[i], confidence, i))

print("Total confusion types:", len(confusion_dict))

In [ ]:
# ========== SELECT REPRESENTATIVE IMAGES FOR QUALITATIVE ANALYSIS ========== #
import shutil

QUAL_DIR = os.path.join(RESULT_DIR, 'qualitative_analysis')
os.makedirs(QUAL_DIR, exist_ok=True)

N_PER_PAIR = 3
print(f"Copying up to {N_PER_PAIR} high-confidence misclassified images per confusion pair...")

for (true_cls, pred_cls), samples in confusion_dict.items():
    samples_sorted = sorted(samples, key=lambda x: -x[1])[:N_PER_PAIR]
    pair_dir       = os.path.join(QUAL_DIR, f'true_{true_cls}__pred_{pred_cls}')
    os.makedirs(pair_dir, exist_ok=True)
    for fp, conf, _ in samples_sorted:
        fname = os.path.basename(fp)
        dst   = os.path.join(pair_dir, f'conf{conf:.3f}_{fname}')
        try:
            shutil.copy2(fp, dst)
        except Exception as e:
            print(f"  [WARN] Could not copy {fname}: {e}")

total_copied = sum(
    len(os.listdir(os.path.join(QUAL_DIR, d)))
    for d in os.listdir(QUAL_DIR)
    if os.path.isdir(os.path.join(QUAL_DIR, d))
)
print(f"Done. {total_copied} images copied to {QUAL_DIR}")

In [ ]:
# ========== SAVE ANALYSIS NOTES ========== #
notes_path = os.path.join(RESULT_DIR, 'analysis_notes.txt')
with open(notes_path, 'w', encoding='utf-8') as f:
    f.write(f"EfficientNetB4 + XGBoost — Qualitative Error Analysis Notes\n")
    f.write(f"{'='*65}\n\n")
    f.write(f"Strategy  : {STRATEGY_LABEL}\n")
    f.write(f"Run       : {SELECTED_RUN} (seed={RANDOM_SEEDS[SELECTED_RUN-1]})\n")
    f.write(f"Date      : {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}\n\n")
    f.write(f"Overall Accuracy : {np.mean(y_true == y_pred):.4f}\n")
    f.write(f"Total Samples    : {len(y_true)}\n")
    f.write(f"Misclassified    : {np.sum(y_true != y_pred)}\n\n")
    f.write(f"Confusion Pairs (True -> Predicted | Count):\n")
    f.write(f"{'-'*65}\n")
    for (tc, pc), smp in sorted(confusion_dict.items(), key=lambda x: -len(x[1])):
        f.write(f"  {tc:<22} -> {pc:<22}  ({len(smp)} samples)\n")
    f.write(f"\nNotes:\n{'.'*65}\n")
    f.write("(Add manual observations here after visual inspection)\n")
print(f"Saved → {notes_path}")

In [ ]:
# ========== ZIP QUALITATIVE ANALYSIS ========== #
import zipfile

qual_zip_path = os.path.join(RESULT_DIR, 'qualitative_analysis.zip')
with zipfile.ZipFile(qual_zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(QUAL_DIR):
        for fname in files:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, RESULT_DIR)
            zf.write(fpath, arcname)
print(f"ZIP saved: {qual_zip_path}  ({os.path.getsize(qual_zip_path) / 1024:.1f} KB)")

## Section 4 — Model Profiling & Final Reports

Comprehensive profiling of the selected run: model parameters, disk size, inference speed (CNN + XGBoost separately), Top-K accuracy, per-class metrics, and a final SUMMARY_REPORT.txt.

In [ ]:
# ========== MODEL SUMMARY & PARAMETERS ========== #
print("="*70)
print("MODEL SUMMARY: EfficientNetB4 Backbone + XGBoost Head")
print("="*70)

# Backbone parameter count
total_params = backbone_model.count_params()
trainable_p  = sum(tf.keras.backend.count_params(w)
                   for w in backbone_model.trainable_weights)
frozen_p     = total_params - trainable_p

print(f"\n[ EfficientNetB4 Backbone ]")
print(f"  Total parameters    : {total_params:,}")
print(f"  Trainable (unfrozen): {trainable_p:,}  ({trainable_p/total_params*100:.1f}%)")
print(f"  Frozen              : {frozen_p:,}  ({frozen_p/total_params*100:.1f}%)")
print(f"  Feature dim output  : 1792  (GlobalAveragePooling2D)")
print(f"  Unfrozen blocks     : {UNFREEZE_BLOCKS}")

# XGBoost info
print(f"\n[ XGBoost Head ]")
print(f"  n_estimators        : {xgb_clf.n_estimators}")
print(f"  max_depth           : {xgb_clf.max_depth}")
print(f"  learning_rate       : {xgb_clf.learning_rate}")
print(f"  input_dim           : 1792")
print(f"  num_class           : {len(class_names)}")
best_iter = getattr(xgb_clf, 'best_iteration', None)
print(f"  early_stopping at   : {best_iter if best_iter is not None else 'N/A'}")
print(f"  actual trees        : {best_iter * len(class_names) if best_iter else 'N/A'}")

print(f"\n[ Top-5 XGBoost Feature Importances (this run) ]")
fi_run   = xgb_clf.feature_importances_
top5_idx = np.argsort(fi_run)[-5:][::-1]
for rank, feat in enumerate(top5_idx):
    print(f"  Rank {rank+1}  GAP-dim {feat:<6}: {fi_run[feat]:.6f}")

In [ ]:
# ========== MODEL SIZE ========== #
import tempfile

# Backbone disk size
tmp_backbone = os.path.join(tempfile.gettempdir(), 'tmp_backbone_size.keras')
backbone_model.save(tmp_backbone)
backbone_mb = os.path.getsize(tmp_backbone) / (1024 ** 2)
os.remove(tmp_backbone)

# XGBoost JSON size
xgb_json_path = os.path.join(RESULT_DIR, 'xgboost_model.json')
xgb_kb = os.path.getsize(xgb_json_path) / 1024 if os.path.exists(xgb_json_path) else 0.0

print("="*55)
print("MODEL DISK SIZE")
print("="*55)
print(f"  EfficientNetB4 backbone : {backbone_mb:.2f} MB  (.keras)")
print(f"  XGBoost model           : {xgb_kb:.2f} KB  (.json)")
print(f"  Total                   : {backbone_mb + xgb_kb/1024:.2f} MB")

In [ ]:
# ========== INFERENCE SPEED (CNN + XGBoost separate timing) ========== #
import time

# Warm-up
_dummy_batch = np.zeros((8, *INPUT_SHAPE), dtype=np.float32)
_ = feature_extractor(_dummy_batch, training=False)
_ = xgb_clf.predict_proba(np.zeros((8, 1792), dtype=np.float32))

SPEED_N         = min(256, len(y_true))
SPEED_BATCH     = 32
n_speed_batches = int(np.ceil(SPEED_N / SPEED_BATCH))

_, _, test_ds_speed, _ = create_tf_datasets(
    DATA_DIR, INPUT_SHAPE, SPEED_BATCH,
    seed=RANDOM_SEEDS[SELECTED_RUN - 1], use_aug=False)
test_speed_iter = iter(test_ds_speed.take(n_speed_batches))

cnn_imgs_all = []
for xb, _ in test_speed_iter:
    cnn_imgs_all.append(xb.numpy())
cnn_imgs_all = np.concatenate(cnn_imgs_all, axis=0)[:SPEED_N]

# --- CNN timing ---
t0 = time.perf_counter()
cnn_feats_speed = feature_extractor(tf.cast(cnn_imgs_all, tf.float32), training=False).numpy()
cnn_time_s = time.perf_counter() - t0

# --- XGBoost timing ---
t0 = time.perf_counter()
_ = xgb_clf.predict_proba(cnn_feats_speed)
xgb_time_s = time.perf_counter() - t0

total_time_s = cnn_time_s + xgb_time_s

print("="*60)
print(f"INFERENCE SPEED  (n={SPEED_N} images, batch={SPEED_BATCH})")
print("="*60)
print(f"  CNN  (EfficientNetB4 feature extraction)")
print(f"    Total    : {cnn_time_s*1000:.1f} ms")
print(f"    Per image: {cnn_time_s/SPEED_N*1000:.3f} ms")
print(f"    FPS      : {SPEED_N/cnn_time_s:.1f}")
print()
print(f"  XGBoost (classification head)")
print(f"    Total    : {xgb_time_s*1000:.1f} ms")
print(f"    Per image: {xgb_time_s/SPEED_N*1000:.3f} ms")
print(f"    FPS      : {SPEED_N/xgb_time_s:.1f}")
print()
print(f"  Combined (CNN + XGBoost)")
print(f"    Total    : {total_time_s*1000:.1f} ms")
print(f"    Per image: {total_time_s/SPEED_N*1000:.3f} ms")
print(f"    FPS      : {SPEED_N/total_time_s:.1f}")
print(f"    CNN/total ratio: {cnn_time_s/total_time_s*100:.1f}%")

In [ ]:
# ========== TOP-K ACCURACY (K = 1, 3, 5) ========== #
# Uses XGBoost predict_proba on pre-extracted test_features

def top_k_accuracy(y_true_arr, proba_arr, k):
    """Return top-k accuracy: true class is within top-k predictions."""
    top_k_preds = np.argsort(-proba_arr, axis=1)[:, :k]
    return np.mean([y_true_arr[i] in top_k_preds[i] for i in range(len(y_true_arr))])

xgb_proba = xgb_clf.predict_proba(test_features)

print("="*50)
print("TOP-K ACCURACY (XGBoost)")
print("="*50)
for k in [1, 3, 5]:
    if k <= len(class_names):
        acc_k = top_k_accuracy(y_true, xgb_proba, k)
        print(f"  Top-{k} Accuracy : {acc_k:.4f}  ({acc_k*100:.2f}%)")

fig, ax = plt.subplots(figsize=(7, 4))
ks    = [k for k in [1, 3, 5] if k <= len(class_names)]
accs  = [top_k_accuracy(y_true, xgb_proba, k) for k in ks]
bars  = ax.bar([f'Top-{k}' for k in ks], accs,
               color=['steelblue', 'orange', 'green'][:len(ks)],
               edgecolor='black', width=0.5)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{acc:.4f}', ha='center', fontweight='bold', fontsize=11)
ax.set_ylim([0, 1.1]); ax.set_ylabel('Accuracy'); ax.grid(axis='y', alpha=0.3)
ax.set_title(f'Top-K Accuracy — EfficientNetB4 + XGBoost  (Run {SELECTED_RUN})',
             fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'topk_accuracy.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Saved → topk_accuracy.png")

In [ ]:
# ========== PER-CLASS METRICS ========== #
from sklearn.metrics import classification_report

report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
report_str  = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print("Per-Class Classification Report:\n")
print(report_str)

per_class_df = pd.DataFrame(report_dict).T
per_class_df = per_class_df.drop(index=['accuracy'], errors='ignore')
per_class_df = per_class_df.round(4)
csv_path = os.path.join(RESULT_DIR, 'per_class_metrics.csv')
per_class_df.to_csv(csv_path)
print(f"Saved → per_class_metrics.csv")

# Bar chart
class_rows = per_class_df.loc[class_names]
fig, axes  = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, ['precision', 'recall', 'f1-score']):
    vals = class_rows[metric].values
    colors = plt.cm.RdYlGn(vals)
    bars = ax.barh(class_names, vals, color=colors, edgecolor='black', height=0.6)
    ax.set_xlim([0, 1.15]); ax.set_xlabel(metric.capitalize())
    ax.set_title(f'Per-Class {metric.capitalize()}', fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f'{v:.3f}', va='center', fontsize=8)

plt.suptitle(f'Per-Class Metrics — EfficientNetB4 + XGBoost  (Run {SELECTED_RUN})',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'per_class_metrics.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Saved → per_class_metrics.png")

In [ ]:
# ========== PREDICTIONS CSV (with Top-3 XGBoost probabilities) ========== #
xgb_proba_all = xgb_clf.predict_proba(test_features)
top3_idx_all  = np.argsort(-xgb_proba_all, axis=1)[:, :3]

pred_rows = []
for i in range(len(y_true)):
    row = {
        'filename'  : test_filenames[i] if i < len(test_filenames) else str(i),
        'true_class': class_names[y_true[i]],
        'pred_class': class_names[y_pred[i]],
        'correct'   : y_true[i] == y_pred[i],
    }
    for rank in range(3):
        ci = top3_idx_all[i, rank]
        row[f'top{rank+1}_class'] = class_names[ci]
        row[f'top{rank+1}_prob']  = round(float(xgb_proba_all[i, ci]), 6)
    pred_rows.append(row)

pred_df  = pd.DataFrame(pred_rows)
csv_path = os.path.join(RESULT_DIR, 'predictions_detail.csv')
pred_df.to_csv(csv_path, index=False)

print(f"predictions_detail.csv  ({len(pred_df)} rows)  saved.")
print(pred_df.head(10).to_string(index=False))

In [ ]:
# ========== COMPREHENSIVE SUMMARY REPORT (Single-Run) ========== #
from sklearn.metrics import (cohen_kappa_score, matthews_corrcoef,
                              balanced_accuracy_score, classification_report)

acc       = np.mean(y_true == y_pred)
kappa     = cohen_kappa_score(y_true, y_pred)
mcc       = matthews_corrcoef(y_true, y_pred)
bal_acc   = balanced_accuracy_score(y_true, y_pred)
report_d  = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
macro_f1  = report_d['macro avg']['f1-score']
macro_p   = report_d['macro avg']['precision']
macro_r   = report_d['macro avg']['recall']

# Timing variables from inference speed cell
try:
    fps = SPEED_N / total_time_s
    ms_img = total_time_s / SPEED_N * 1000
    cnn_ms = cnn_time_s / SPEED_N * 1000
    xgb_ms = xgb_time_s / SPEED_N * 1000
except NameError:
    fps = cnn_ms = xgb_ms = ms_img = float('nan')

topk_accs = {}
for k in [1, 3, 5]:
    if k <= len(class_names):
        topk_accs[k] = top_k_accuracy(y_true, xgb_clf.predict_proba(test_features), k)
    else:
        topk_accs[k] = float('nan')

report_txt = f"""
EfficientNetB4 + XGBoost — Complete Summary Report
{'='*65}
Strategy   : {STRATEGY_LABEL}
Run        : {SELECTED_RUN}  (seed={RANDOM_SEEDS[SELECTED_RUN-1]})
Date       : {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}
Result Dir : {RESULT_DIR}

[ Architecture ]
  Backbone       : EfficientNetB4 (pretrained ImageNet)
  Unfrozen blocks: {UNFREEZE_BLOCKS}
  Feature dim    : 1792  (GlobalAveragePooling2D)
  Classifier     : XGBoost  (n_estimators={xgb_clf.n_estimators}, max_depth={xgb_clf.max_depth})
  Input shape    : {INPUT_SHAPE}
  Num classes    : {len(class_names)}
  Classes        : {class_names}

[ Classification Metrics ]
  Accuracy           : {acc:.4f}
  Macro F1           : {macro_f1:.4f}
  Macro Precision    : {macro_p:.4f}
  Macro Recall       : {macro_r:.4f}
  Cohen Kappa        : {kappa:.4f}
  Matthews Corrcoef  : {mcc:.4f}
  Balanced Accuracy  : {bal_acc:.4f}
  Top-1 Accuracy     : {topk_accs.get(1, float('nan')):.4f}
  Top-3 Accuracy     : {topk_accs.get(3, float('nan')):.4f}
  Top-5 Accuracy     : {topk_accs.get(5, float('nan')):.4f}

[ Inference Speed ]
  CNN  (feature extraction): {cnn_ms:.3f} ms/image
  XGBoost (classification) : {xgb_ms:.3f} ms/image
  Combined                 : {ms_img:.3f} ms/image  ({fps:.1f} FPS)

[ XGBoost Training Config ]
  {XGB_PARAMS}

{'='*65}
"""

report_path = os.path.join(RESULT_DIR, 'SUMMARY_REPORT.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_txt)
print(report_txt)
print(f"Saved → SUMMARY_REPORT.txt")

In [ ]:
# ========== LIST ALL REPORT FILES ========== #
import glob

print(f"Files in: {RESULT_DIR}\n")
all_files = sorted(glob.glob(os.path.join(RESULT_DIR, '**', '*'), recursive=True))
all_files = [f for f in all_files if os.path.isfile(f)]

total_size = 0
rows = []
for fp in all_files:
    size   = os.path.getsize(fp)
    total_size += size
    rel    = os.path.relpath(fp, RESULT_DIR)
    size_s = f'{size/1024:.1f} KB' if size < 1024**2 else f'{size/1024**2:.2f} MB'
    rows.append({'file': rel, 'size': size_s})
    print(f"  {rel:<60} {size_s}")

print(f"\nTotal: {len(all_files)} files | {total_size / 1024**2:.2f} MB")

In [ ]:
# ========== ZIP ALL REPORTS ========== #
import zipfile

final_zip = os.path.join('/kaggle/working', 'EfficientNetB4_XGBoost_Complete_Report.zip')
with zipfile.ZipFile(final_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fp in all_files:
        arcname = os.path.relpath(fp, '/kaggle/working')
        zf.write(fp, arcname)

zip_mb = os.path.getsize(final_zip) / (1024 ** 2)
print(f"Final ZIP saved: {final_zip}  ({zip_mb:.2f} MB)")
print(f"  Contains {len(all_files)} files from {RESULT_DIR}")
print("\nNotebook execution complete.")
print(f"Strategy  : {STRATEGY_LABEL}")
print(f"RESULT DIR: {RESULT_DIR}")